In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import gradio as gr
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ============================================================
# HELPERS
# ============================================================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9äöüß\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalize_strategy(x):
    x = normalize_text(x).replace("-", "_").replace(" ", "_")
    mapping = {
        "standard": "standard",
        "co_creation": "co_creation",
        "cocreation": "co_creation",
        "limited_edition": "limited_edition",
        "limited": "limited_edition",
    }
    return mapping.get(x, "standard")


def token_similarity(a, b):
    a_tokens = set(normalize_text(a).split())
    b_tokens = set(normalize_text(b).split())
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def scenario_bounds(base_value, confidence):
    uncertainty = 0.35 - (confidence - 0.30) * 0.25
    uncertainty = float(np.clip(uncertainty, 0.12, 0.35))
    low = int(round(base_value * (1 - uncertainty)))
    high = int(round(base_value * (1 + uncertainty)))
    return low, int(round(base_value)), high


def future_month_choices(n=24):
    start = pd.Timestamp.today().replace(day=1) + pd.offsets.MonthBegin(1)
    return [(start + pd.DateOffset(months=i)).strftime("%Y-%m") for i in range(n)]


# ============================================================
# RAW DATA LOADER (NO HARDCODED HISTORICAL DEFINITIONS)
# ============================================================
def load_raw_data():
    orders_path = Path("data/raw/orders.csv")
    launches_path = Path("data/raw/launched_product_details.csv")
    sale_times_path = Path("data/raw/sale_times.csv")

    for p in [orders_path, launches_path, sale_times_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing raw file: {p}")

    orders = pd.read_csv(orders_path, low_memory=False)
    orders.columns = orders.columns.str.strip()

    launches = pd.read_csv(launches_path, sep=";", low_memory=False)
    launches.columns = launches.columns.str.strip()

    sale_times = pd.read_csv(sale_times_path, sep=";", low_memory=False)
    sale_times.columns = sale_times.columns.str.strip()

    return orders, launches, sale_times


# Load once; pipeline consumes this cache on every UI run.
RAW_ORDERS, RAW_LAUNCHES, RAW_SALE_TIMES = load_raw_data()


# ============================================================
# MAIN PIPELINE (INPUT-DRIVEN)
# ============================================================
def run_pipeline(new_launch_input):
    orders = RAW_ORDERS.copy()
    launches = RAW_LAUNCHES.copy()
    sale_times = RAW_SALE_TIMES.copy()

    required_order_cols = [
        "customer_nr", "order_id", "sku", "date", "product", "flavour", "price", "quantity", "net_revenue", "customer_status"
    ]
    missing_order = [c for c in required_order_cols if c not in orders.columns]
    if missing_order:
        raise ValueError(f"orders.csv missing columns: {missing_order}")

    required_launch_cols = [
        "sku", "launch_date", "product", "flavour", "product_form", "launch_strategy_type",
        "Target Group", "Product Use Case / What it is about", "uvp"
    ]
    missing_launch = [c for c in required_launch_cols if c not in launches.columns]
    if missing_launch:
        raise ValueError(f"launched_product_details.csv missing columns: {missing_launch}")

    # Clean orders
    orders["date"] = pd.to_datetime(orders["date"], errors="coerce")
    orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
    orders["price"] = pd.to_numeric(orders["price"], errors="coerce")
    orders["net_revenue"] = pd.to_numeric(orders["net_revenue"], errors="coerce")
    orders = orders.dropna(subset=["customer_nr", "order_id", "sku", "date"])
    orders = orders[orders["quantity"].fillna(0) > 0].copy()

    orders["product_norm"] = orders["product"].apply(normalize_text)
    orders["flavour_norm"] = orders["flavour"].apply(normalize_text)
    orders["customer_status_norm"] = orders["customer_status"].astype(str).str.upper()

    # Clean launches
    launches["launch_date"] = pd.to_datetime(launches["launch_date"], errors="coerce")
    launches["uvp"] = launches["uvp"].astype(str).str.replace("€", "", regex=False).str.replace(",", ".", regex=False)
    launches["uvp"] = pd.to_numeric(launches["uvp"], errors="coerce")
    launches = launches.dropna(subset=["sku", "launch_date"])

    launches["target_group_norm"] = launches["Target Group"].apply(normalize_text)
    launches["use_case_norm"] = launches["Product Use Case / What it is about"].apply(normalize_text)
    launches["flavour_norm"] = launches["flavour"].apply(normalize_text)
    launches["product_form_norm"] = launches["product_form"].apply(normalize_text)
    launches["strategy_norm"] = launches["launch_strategy_type"].apply(normalize_strategy)

    # Clean sale times and add order flag
    sale_times["start_d"] = pd.to_datetime(sale_times["start_d"], errors="coerce")
    sale_times["end_d"] = pd.to_datetime(sale_times["end_d"], errors="coerce")
    sale_times = sale_times.dropna(subset=["start_d", "end_d"])

    orders["is_sale_period"] = 0
    for _, s in sale_times.iterrows():
        mask = (orders["date"] >= s["start_d"]) & (orders["date"] <= s["end_d"])
        orders.loc[mask, "is_sale_period"] = 1

    latest_date = orders["date"].max()

    # Input normalization
    new_tg = normalize_text(new_launch_input["target_group"])
    new_uc = normalize_text(new_launch_input["product_use_case"])
    new_fl = normalize_text(new_launch_input["flavour"])
    new_pf = normalize_text(new_launch_input["product_type"])
    new_st = normalize_strategy(new_launch_input["launch_strategy_type"])
    new_uvp = float(new_launch_input["uvp"])

    # Launch similarity
    launches["sim_target_group"] = launches["target_group_norm"].apply(lambda x: token_similarity(x, new_tg))
    launches["sim_use_case"] = launches["use_case_norm"].apply(lambda x: token_similarity(x, new_uc))
    launches["sim_flavour"] = launches["flavour_norm"].apply(lambda x: token_similarity(x, new_fl))
    launches["sim_product_form"] = launches["product_form_norm"].apply(lambda x: token_similarity(x, new_pf))
    launches["sim_strategy"] = (launches["strategy_norm"] == new_st).astype(float)
    launches["sim_price"] = launches["uvp"].apply(
        lambda p: 0.5 if pd.isna(p) or p <= 0 or new_uvp <= 0 else min(new_uvp, p) / max(new_uvp, p)
    )

    launches["launch_similarity"] = (
        0.25 * launches["sim_target_group"]
        + 0.25 * launches["sim_use_case"]
        + 0.15 * launches["sim_flavour"]
        + 0.15 * launches["sim_product_form"]
        + 0.10 * launches["sim_strategy"]
        + 0.10 * launches["sim_price"]
    )

    sim_threshold = float(np.quantile(launches["launch_similarity"], 0.75))
    selected_launches = launches[launches["launch_similarity"] >= sim_threshold].copy()
    if selected_launches.empty:
        selected_launches = launches.nlargest(10, "launch_similarity").copy()

    # Customer features
    input_keywords = set(
        new_tg.split() + new_uc.split() + new_fl.split() + new_pf.split() + normalize_text(new_launch_input["flavour_type"]).split()
    )
    input_keywords = {k for k in input_keywords if len(k) > 2}

    def overlap_score(text):
        tokens = set(normalize_text(text).split())
        if not tokens or not input_keywords:
            return 0.0
        return len(tokens & input_keywords) / len(tokens | input_keywords)

    orders["text_overlap"] = (orders["product_norm"].fillna("") + " " + orders["flavour_norm"].fillna("")).apply(overlap_score)

    customer_features = (
        orders.groupby("customer_nr")
        .agg(
            recency_days=("date", lambda s: (latest_date - s.max()).days),
            frequency_orders=("order_id", "nunique"),
            monetary=("net_revenue", "sum"),
            avg_price=("price", "mean"),
            avg_qty=("quantity", "mean"),
            sale_share=("is_sale_period", "mean"),
            overlap_mean=("text_overlap", "mean"),
            overlap_max=("text_overlap", "max"),
            recent_90d_orders=("date", lambda s: int((s >= (latest_date - pd.Timedelta(days=90))).sum())),
            is_new_customer=("customer_status_norm", lambda s: int((s == "NEW").any())),
        )
        .reset_index()
    )

    # Labels from historical similar launch buyers
    launch_buyer_rows = []
    for _, lr in selected_launches.iterrows():
        sku = lr["sku"]
        start = lr["launch_date"]
        end = start + pd.Timedelta(days=41)
        buyers = orders[(orders["sku"] == sku) & (orders["date"] >= start) & (orders["date"] <= end)]["customer_nr"].dropna().unique()
        for c in buyers:
            launch_buyer_rows.append((c, sku, float(lr["launch_similarity"])))

    if not launch_buyer_rows:
        raise ValueError("No historical launch buyers found for similar launches. Try broader launch input.")

    launch_buyer_df = pd.DataFrame(launch_buyer_rows, columns=["customer_nr", "sku", "similarity"])
    label_df = launch_buyer_df.groupby("customer_nr")["similarity"].max().reset_index().rename(columns={"similarity": "max_sim_bought"})
    label_df["target_buy_new_launch"] = (label_df["max_sim_bought"] > 0).astype(int)

    model_df = customer_features.merge(label_df[["customer_nr", "target_buy_new_launch"]], on="customer_nr", how="left")
    model_df["target_buy_new_launch"] = model_df["target_buy_new_launch"].fillna(0).astype(int)

    feature_cols = [
        "recency_days", "frequency_orders", "monetary", "avg_price", "avg_qty",
        "sale_share", "overlap_mean", "overlap_max", "recent_90d_orders", "is_new_customer"
    ]

    X = model_df[feature_cols].fillna(0.0)
    y = model_df["target_buy_new_launch"].astype(int)

    if y.nunique() < 2:
        raise ValueError("Not enough class diversity to train models for this input.")

    # Multi-model propensity
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_all_scaled = scaler.transform(X)

    models = {
        "log_reg": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "random_forest": RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42, class_weight="balanced_subsample"),
        "grad_boost": GradientBoostingClassifier(random_state=42),
    }

    auc_scores = {}
    model_preds = {}

    for name, model in models.items():
        if name == "log_reg":
            model.fit(X_train_scaled, y_train)
            p_test = model.predict_proba(X_test_scaled)[:, 1]
            p_all = model.predict_proba(X_all_scaled)[:, 1]
        else:
            model.fit(X_train, y_train)
            p_test = model.predict_proba(X_test)[:, 1]
            p_all = model.predict_proba(X)[:, 1]

        auc_scores[name] = roc_auc_score(y_test, p_test)
        model_preds[name] = p_all

    auc_df = pd.DataFrame([{"Model": k, "AUC": round(v, 4)} for k, v in auc_scores.items()]).sort_values("AUC", ascending=False)

    auc_sum = sum(auc_scores.values())
    weights = {k: (v / auc_sum if auc_sum > 0 else 1.0 / len(auc_scores)) for k, v in auc_scores.items()}

    ensemble_ml = np.zeros(len(model_df), dtype=float)
    for k in models.keys():
        ensemble_ml += weights[k] * model_preds[k]

    baseline = (
        0.30 * model_df["overlap_max"].values
        + 0.20 * model_df["overlap_mean"].values
        + 0.15 * np.tanh(model_df["frequency_orders"].values / 8)
        + 0.15 * np.tanh(model_df["recent_90d_orders"].values / 4)
        + 0.10 * (1 - np.tanh(model_df["recency_days"].values / 120))
        + 0.10 * model_df["sale_share"].fillna(0).values
    )
    baseline = np.clip(baseline, 0, 1)

    model_df["score_ml_ensemble"] = ensemble_ml
    model_df["score_baseline"] = baseline
    model_df["score_final"] = 0.85 * model_df["score_ml_ensemble"] + 0.15 * model_df["score_baseline"]

    # Scenario forecasts
    expected_buyers_6w = float(model_df["score_final"].sum())
    expected_buyers_1w = expected_buyers_6w * 0.42

    units_per_order_med = float(max(1.0, orders["quantity"].median()))
    base_units_1w = expected_buyers_1w * units_per_order_med
    base_units_6w = expected_buyers_6w * units_per_order_med

    weighted_new_ratio = float((model_df["score_final"] * model_df["is_new_customer"]).sum() / max(model_df["score_final"].sum(), 1e-9))
    base_nc_1w = expected_buyers_1w * weighted_new_ratio
    base_nc_6w = expected_buyers_6w * weighted_new_ratio

    confidence = float(np.clip(np.mean(list(auc_scores.values())), 0.30, 0.90))

    u1_low, u1_base, u1_high = scenario_bounds(base_units_1w, confidence)
    u6_low, u6_base, u6_high = scenario_bounds(base_units_6w, confidence)
    n1_low, n1_base, n1_high = scenario_bounds(base_nc_1w, confidence)
    n6_low, n6_base, n6_high = scenario_bounds(base_nc_6w, confidence)

    scenario_df = pd.DataFrame([
        {"Metric": "First week units", "Worst": u1_low, "Base": u1_base, "Best": u1_high},
        {"Metric": "First 6 week units", "Worst": u6_low, "Base": u6_base, "Best": u6_high},
        {"Metric": "First week new customers", "Worst": n1_low, "Base": n1_base, "Best": n1_high},
        {"Metric": "First 6 week new customers", "Worst": n6_low, "Base": n6_base, "Best": n6_high},
    ])

    top_customers = model_df.sort_values("score_final", ascending=False).head(50).copy()
    top_customers["Rank"] = np.arange(1, len(top_customers) + 1)
    top_customers["Campaign Priority"] = pd.cut(
        top_customers["score_final"],
        bins=[-0.01, 0.25, 0.50, 0.70, 1.01],
        labels=["Low", "Medium", "High", "Very High"],
    )

    top_cols = [
        "Rank", "customer_nr", "score_final", "score_ml_ensemble", "score_baseline",
        "is_new_customer", "frequency_orders", "recency_days", "Campaign Priority"
    ]

    explanation = (
        "1) Input from UI converted to launch profile\n"
        "2) Similar historical launches selected from raw launch data\n"
        "3) Customer features built from raw orders + sale windows\n"
        "4) Three ML propensity models trained and AUC-weighted ensemble built\n"
        "5) Scenario outputs derived from expected buyers and confidence uncertainty"
    )

    return auc_df, scenario_df, top_customers[top_cols], f"Confidence: {confidence:.3f}", explanation


# ============================================================
# UI: USER INPUT DRIVES NEW LAUNCH PROFILE
# ============================================================
def run_from_ui(
    target_group,
    product_use_case,
    launch_month_year,
    launch_strategy_type,
    uvp,
    flavour,
    flavour_type,
    product_type,
):
    payload = {
        "target_group": target_group,
        "product_use_case": product_use_case,
        "launch_month_year": launch_month_year,
        "launch_strategy_type": launch_strategy_type,
        "uvp": float(uvp),
        "flavour": flavour,
        "flavour_type": flavour_type,
        "product_type": product_type,
    }
    return run_pipeline(payload)


month_choices = future_month_choices(24)

with gr.Blocks(title="Launch Buyer Propensity") as demo:
    gr.Markdown("## New Launch Input (UI-driven)")

    with gr.Row():
        inp_target_group = gr.Textbox(label="Target group", value="Health-conscious adults")
        inp_use_case = gr.Textbox(label="Product use case", value="Supports gut microbiota and daily digestive balance")

    with gr.Row():
        inp_month_year = gr.Dropdown(label="Launch month-year", choices=month_choices, value=month_choices[0])
        inp_strategy = gr.Dropdown(label="Launch strategy type", choices=["standard", "co_creation", "limited_edition"], value="standard")

    with gr.Row():
        inp_uvp = gr.Number(label="UVP", value=49.9, minimum=0.1)
        inp_flavour = gr.Textbox(label="Flavour", value="Lemon")

    with gr.Row():
        inp_flavour_type = gr.Dropdown(label="Flavour type", choices=["sweet", "sour", "bitter", "neutral", "no_flavour"], value="sour")
        inp_product_type = gr.Dropdown(label="Product type", choices=["Drinking Powder", "Capsules", "Spray", "Oil"], value="Drinking Powder")

    btn = gr.Button("Run Forecast", variant="primary")

    out_conf = gr.Textbox(label="Confidence")
    out_explain = gr.Textbox(label="How result is produced", lines=6)
    out_auc = gr.DataFrame(label="Model quality (AUC)")
    out_scenario = gr.DataFrame(label="Scenario forecasts")
    out_top = gr.DataFrame(label="Top 50 customers most likely to buy")

    btn.click(
        fn=run_from_ui,
        inputs=[
            inp_target_group,
            inp_use_case,
            inp_month_year,
            inp_strategy,
            inp_uvp,
            inp_flavour,
            inp_flavour_type,
            inp_product_type,
        ],
        outputs=[out_auc, out_scenario, out_top, out_conf, out_explain],
    )

demo

Saving forecast_trail_1.csv to forecast_trail_1 (1).csv
